In [1]:
# Importing modules
import pandas as pd
import numpy as np
from src.utils.smiles2morganfp import MorganFP
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Loading ESOL data
esol_data_train = pd.read_csv("data/train/ESOL.csv")
esol_data_val = pd.read_csv("data/val/ESOL.csv")

# Loading Lipophilicity data
lipophilicity_data_train = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_data_val = pd.read_csv("data/val/Lipophilicity.csv")

# Loading FreeSolv data
freesolv_data_train = pd.read_csv("data/train/FreeSolv.csv")
freesolv_data_val = pd.read_csv("data/val/FreeSolv.csv")

# Loading B3DB data
b3db_data_train = pd.read_csv("data/train/B3DB.csv")
b3db_data_val = pd.read_csv("data/val/B3DB.csv")

# Loading RT data
rt_data_train = pd.read_csv("data/train/RT.csv")
rt_data_val = pd.read_csv("data/val/RT.csv")

In [3]:
# Function to run analysis
def RunML(data, dataName, modelName):

	train_data = data["train"]
	val_data = data["val"]

	# Generate FP
	fp_train = MorganFP(train_data["smiles"], bits=1024)
	fp_train["smiles"] = fp_train.index
	fp_train = fp_train.merge(train_data, on="smiles")
	fp_val = MorganFP(val_data["smiles"], bits=1024)
	fp_val["smiles"] = fp_val.index
	fp_val = fp_val.merge(val_data, on="smiles")
    
	# Feature Matrix (molecular fingerprint)
	X_train = fp_train.drop(["smiles", "target"], axis=1)
	X_val = fp_val.drop(["smiles", "target"], axis=1)

	# Target labels
	y_train = fp_train["target"]
	y_val = fp_val["target"]

	# Model
	model = GridSearchCV(model_dict[modelName], params_dict[modelName], cv=None, scoring="neg_root_mean_squared_error")

	# Model training
	model = model.fit(X_train.to_numpy(), y_train)

	# Model testing
	y_pred = model.predict(X_val.to_numpy())

	# Calculate MAE
	mae = mean_absolute_error(y_val, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_val, y_pred)

	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
            "Model Params":[model.best_params_],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)]})

In [4]:
# Function for model selection and hyperparameter tuning
def search_hyperparams(dataName):
    temp_out = []
    # Hyperparameter tuning loop
    for modelName in model_dict.keys():
        temp_out.append(RunML(data_dict[dataName], dataName, modelName))

    # Saving results
    final_df = pd.concat(temp_out, ignore_index=True)
    final_df.to_csv(f"results/Output_Hyperparameter_Optimization_ML_{dataName}.csv", quoting=False, index=False)

In [5]:
# Models
model_dict = {
    "LR": LinearRegression(), 
    "SVM": SVR(),
    "RF": RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=16)
}

# Params
params_dict = {
    'LR': {
        'fit_intercept': [True],
        'positive': [False]
    },

    'SVM': {
        'kernel': ['rbf'],
        'C': [0.1, 1, 10],
        'gamma': ['scale'],
        'epsilon': [0.05]
    },

    'RF': {
        'n_estimators': [100, 200],
        'max_depth': [None, 10],
        'min_samples_split': [4],
        'min_samples_leaf': [5],
        'max_features': ['sqrt']
    },

    'XGB': {
        'n_estimators': [50, 100],
        'max_depth': [3, 6],
        'learning_rate': [0.01, 0.001],
        'subsample': [0.8],
        'colsample_bytree': [0.8],
        'alpha': [0]
    }
}

# Data
data_dict = {
    "ESOL": {"train": esol_data_train, "val":esol_data_val}, 
    "Lipophilicity": {"train":lipophilicity_data_train, "val":lipophilicity_data_val},
    "FreeSolv": {"train":freesolv_data_train, "val":freesolv_data_val},
    "RT": {"train":rt_data_train, "val":rt_data_val},
    "B3DB": {"train":b3db_data_train, "val": b3db_data_val}
}

In [6]:
# Params Optim for ESOL
search_hyperparams("ESOL")

# Params Optim for Lipophilicity
search_hyperparams("Lipophilicity")

# Params Optim for FreeSolv
search_hyperparams("FreeSolv")

# Params Optim for RT
search_hyperparams("RT")

# Params Optim for B3DB
search_hyperparams("B3DB")